Análise de E-Commerce - Dataset: Olist

Neste projeto exploro um dataset real de e-commerce brasileiro para responder perguntas de negócio usando Python e SQL.

As perguntas são:

    - Quantos pedidos foram feitos e qual foi a receita total?
    - Qual a média de avaliação dos clientes?
    - Quais são os meses com mais vendas?   
    - Quais estados compram mais?
    - Qual a taxa de atraso nas entregas?


1. Importando as bibliotecas

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
import pandas as pd
import sqlite3

print("Bibliotecas importadas com sucesso")

Bibliotecas importadas com sucesso


2. Carregando dados
    - Apenas tabelas que serão usadas nesta análise

In [30]:
orders    = pd.read_csv("data/raw/olist_orders_dataset.csv")
items     = pd.read_csv("data/raw/olist_order_items_dataset.csv")
reviews   = pd.read_csv("data/raw/olist_order_reviews_dataset.csv")
customers = pd.read_csv("data/raw/olist_customers_dataset.csv")

print(f"Pedidos:    {len(orders):,} linhas")
print(f"Itens:      {len(items):,} linhas")
print(f"Avaliações: {len(reviews):,} linhas")
print(f"Clientes:   {len(customers):,} linhas")


Pedidos:    99,441 linhas
Itens:      112,650 linhas
Avaliações: 99,224 linhas
Clientes:   99,441 linhas


3. Entendendo o conteúdo de cada tabela

In [31]:
# Exibindo as primeiras linhas do DataFrame de pedidos
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [32]:
# Exibindo um resumo do DataFrame de pedidos
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [33]:
# Contando quantas vezes cada status de pedido aparece
orders['order_status'].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


4. Limpeza dos dados
    - Vou converter as datas, transformar as strings de data em objetos datatime e criar colunas úteis para a análise.

In [34]:
# Converter colunas de data para o formato datetime
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

# Criar coluna de ano/mes para análise temporal
orders['ano_mes'] = orders['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Calcular dias de entrega
orders['dias_entrega'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

# Marcar entregas atrasadas: 1 = atrasada, 0 = no prazo
orders['entrega_atrasada'] = (orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']).astype(int)

print("Análise inicial concluída com sucesso")
print(f"Tempo médio de entrega: {orders['dias_entrega'].mean():.1f} dias")


Análise inicial concluída com sucesso
Tempo médio de entrega: 12.1 dias


In [35]:
# calcular valor total de cada item (produto + frete)
items['valor_total'] = items['price'] + items['freight_value']

print(f"receita total: R$ {items['valor_total'].sum():,.2f}")
print(f"ticket médio por item: R$ {items['price'].mean():,.2f}")

receita total: R$ 15,843,553.24
ticket médio por item: R$ 120.65


5. Análise das principais métricas de negócio (KPIs)


In [36]:
# Filtrar só pedidos entregues
entregues = orders[orders['order_status'] == 'delivered']

# Juntar pedidos com itens para calcular receita
pedidos_com_valor = entregues.merge(items, on= 'order_id', how='left')

# calcular KPIs
total_pedidos = entregues['order_id'].nunique()
receita_total = pedidos_com_valor['valor_total'].sum()
ticket_medio = pedidos_com_valor.groupby('order_id')['valor_total'].sum().mean()
nota_media = reviews['review_score'].mean() * 100
taxa_atraso = entregues['entrega_atrasada'].mean() * 100

print("=" * 40)
print(" KPIs DO NEGÓCIO")
print("=" * 40)
print(f" Total de pedidos entregues : {total_pedidos:,}")
print(f" Receita total              : R$ {receita_total:,.2f}")
print(f" Ticket médio por pedido    : R$ {ticket_medio:.2f}")
print(f" Nota média dos clientes    : {nota_media:.2f} / 5.0")
print(f" Taxa de atraso             : {taxa_atraso:.1f}%")
print("=" * 40)

 KPIs DO NEGÓCIO
 Total de pedidos entregues : 96,478
 Receita total              : R$ 15,419,773.75
 Ticket médio por pedido    : R$ 159.83
 Nota média dos clientes    : 408.64 / 5.0
 Taxa de atraso             : 8.1%


6. Criando o Banco de dados
    - Dados limpos e salvos em um banco de dados para fazer consultas em SQL.

In [37]:
conn = sqlite3.connect('olist.db')

orders.to_sql('orders', conn, if_exists='replace', index=False)
items.to_sql('items', conn, if_exists='replace', index=False)
reviews.to_sql('reviews', conn, if_exists='replace', index=False)
customers.to_sql('customers', conn, if_exists='replace', index=False)

print("Dados exportados para o banco de dados SQLite com sucesso")

Dados exportados para o banco de dados SQLite com sucesso


7. Consultas SQL

Pergunta 1: Quais meses tiveram mais vendas?

In [38]:
vendas_por_mes = pd.read_sql("""
    SELECT
        ano_mes,
        COUNT(DISTINCT order_id) AS total_pedidos
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY ano_mes
    ORDER BY ano_mes
""", conn)

vendas_por_mes.tail(12)

,ano_mes,total_pedidos
11,2017-09,4150
12,2017-10,4478
13,2017-11,7289
14,2017-12,5513
15,2018-01,7069
16,2018-02,6555
17,2018-03,7003
18,2018-04,6798
19,2018-05,6749
20,2018-06,6099


Pergunta 2: Quais estados compraram mais produtos?

In [39]:
pedidos_por_estado = pd.read_sql("""
    SELECT
        c.customer_state AS estado,
        COUNT(DISTINCT o.order_id) AS total_pedidos
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY estado
    ORDER BY total_pedidos DESC
    LIMIT 10
""", conn)

print(pedidos_por_estado)

  estado  total_pedidos
0     SP          40501
1     RJ          12350
2     MG          11354
3     RS           5345
4     PR           4923
5     SC           3546
6     BA           3256
7     DF           2080
8     ES           1995
9     GO           1957


Pergunta 3: Qual é a taxa de atraso por estado?

In [40]:
# Ordenado pela porcentagem de atraso em ordem decrescente e limitado aos 10 primeiros estados
atraso_por_estado = pd.read_sql("""
    SELECT
        c.customer_state                                      AS estado,
        COUNT(*)                                              AS total_pedidos,
        SUM(o.entrega_atrasada)                               AS atrasados,
        ROUND(100.0 * SUM(o.entrega_atrasada) / COUNT(*), 1) AS pct_atraso
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.dias_entrega IS NOT NULL
    GROUP BY estado
    HAVING total_pedidos > 100
    ORDER BY pct_atraso DESC
    LIMIT 10
""", conn)

atraso_por_estado


,estado,total_pedidos,atrasados,pct_atraso
0,AL,397,95,23.9
1,MA,717,141,19.7
2,PI,476,76,16.0
3,CE,1279,196,15.3
4,SE,335,51,15.2
5,BA,3256,457,14.0
6,RJ,12350,1664,13.5
7,TO,274,35,12.8
8,PA,946,117,12.4
9,ES,1995,244,12.2


Pergunta 4: Quais são as avaliações dos clientes?

In [41]:
avaliacoes = pd.read_sql("""
    SELECT
        review_score                                        AS nota,
        COUNT(*)                                            AS total,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 1)  AS pct
    FROM reviews
    GROUP BY nota
    ORDER BY nota
""", conn)

avaliacoes

,nota,total,pct
0,1,11424,11.5
1,2,3151,3.2
2,3,8179,8.2
3,4,19142,19.3
4,5,57328,57.8


In [42]:
mediana_avaliacoes = reviews['review_score'].median()
print(f"A mediana da nota das avaliações é: {mediana_avaliacoes:.2f}")

A mediana da nota das avaliações é: 5.00


Este código calcula a mediana da coluna `review_score` do DataFrame `reviews` e exibe o resultado formatado para duas casas decimais.

8. Salvando os resultados como CSV para criação do dashboard no Power BI

In [43]:
import os
os.makedirs("data/processed", exist_ok=True)

orders.to_csv("data/processed/orders_clean.csv", index=False)
items.to_csv("data/processed/items_clean.csv", index=False)
reviews.to_csv("data/processed/reviews_clean.csv", index=False)
customers.to_csv("data/processed/customers_clean.csv", index=False)

vendas_por_mes.to_csv("data/processed/vendas_por_mes.csv", index=False)
pedidos_por_estado.to_csv("data/processed/pedidos_por_estado.csv", index=False)
atraso_por_estado.to_csv("data/processed/atraso_por_estado.csv", index=False)
avaliacoes.to_csv("data/processed/avaliacoes.csv", index=False)

print("Completo")


#Banco de dados encerrado
conn.close()

Completo


9. **Conclusões**
  - O dataset mostrou o total de **96,478** pedidos entregues entre 2016 e 2018.
  - A receita total foi de **R$ 15,419,773.75.**
  - O mês que mais vendeu foi em **novembro/2017**, pode ter relação com a Black Friday.
  - **Alagoas (AL) e Maranhão(MA)** tiveram as maiores taxas de atraso nos pedidos: **23.9% e 19.7%**
  - A **nota mediana** dos clientes foi de **5/5** | Maioria dos clientes satisfeitos: **57.8%.**

### Download da pasta 'processed'

Execute as células a seguir para baixar a pasta dos dados limpors - `data/processed`


In [44]:
import shutil

# Compactando a pasta 'processed'
output_filename = 'processed_data'
shutil.make_archive(output_filename, 'zip', 'data/processed')
print(f"Folder 'data/processed' compressed to {output_filename}.zip")

Folder 'data/processed' compressed to processed_data.zip


In [45]:
from google.colab import files

# Fornecer um link de download para o arquivo compactado
files.download('processed_data.zip')

print("Seu download deve começar em breve.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Seu download deve começar em breve.


---
*Projeto desenvolvido por Camila Rodrigues Mota*  
*Ferramentas: Python (Pandas) • SQL (SQLite) • Power BI*  
*Dataset: [Brazilian E-Commerce Public Dataset — Olist/Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)*